# Phase 2 — Step 3: Clustering
**K-Means · DBSCAN · Hierarchical**

| | |
|---|---|
| Dataset | Home Credit Default Risk |
| Input | `features_pca50.csv` (ROW_ID + PC1–PC50) |
| K optimal | **5** (Elbow Method) |
| PCA komponen | **30** (90% variance) |

---
**Urutan cell:**
1. Cek environment Kaggle
2. Load data
3. K-Means (MiniBatchKMeans)
4. DBSCAN
5. Hierarchical + Dendrogram
6. Visualisasi
7. Simpan & download hasil

---
## Cell 1 — Cek Environment Kaggle

In [ ]:
# ── CEK ENVIRONMENT ──────────────────────────────────────────────
# Di Kaggle, semua library data science sudah pre-installed.
# Tidak perlu install apapun — langsung import.

import subprocess
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import MiniBatchKMeans, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
import time
import os

print('=' * 55)
print('CEK ENVIRONMENT KAGGLE')
print('=' * 55)

# Cek GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    for line in result.stdout.split('\n'):
        if any(g in line for g in ['Tesla', 'T4', 'P100', 'A100', 'V100']):
            print(f'  GPU : {line.strip()}')
            break
    print('  ✅ GPU tersedia (tidak dipakai sklearn, tapi RAM lebih besar)')
else:
    print('  ⚠️  Tidak ada GPU — pastikan Accelerator di-set ke GPU di Settings kanan')

# Cek RAM
ram_gb = psutil.virtual_memory().total / 1e9
print(f'  RAM             : {ram_gb:.1f} GB')
print(f'  scikit-learn    : {sklearn.__version__}')
print(f'  numpy           : {np.__version__}')
print(f'  pandas          : {pd.__version__}')

# Cek lokasi file input di Kaggle
# Setelah upload dataset, file ada di /kaggle/input/<nama-dataset>/
INPUT_DIR = '/kaggle/input'
print(f'\n  Dataset tersedia di {INPUT_DIR}:')
for folder in os.listdir(INPUT_DIR):
    files = os.listdir(os.path.join(INPUT_DIR, folder))
    for f in files:
        size_mb = os.path.getsize(os.path.join(INPUT_DIR, folder, f)) / 1e6
        print(f'    /kaggle/input/{folder}/{f}  ({size_mb:.1f} MB)')

print()
print('✅ Environment siap! Lanjut ke Cell 2.')

---
## Cell 2 — Load Data

In [ ]:
# ── KONFIGURASI ──────────────────────────────────────────────────
# Ganti <nama-dataset> dengan nama folder yang muncul di Cell 1
DATASET_NAME = '<nama-dataset>'   # ← GANTI INI

FILE_PCA50 = f'/kaggle/input/{DATASET_NAME}/features_pca50.csv'
FILE_PCA2  = f'/kaggle/input/{DATASET_NAME}/features_pca2.csv'

K_OPTIMAL    = 5    # hasil Elbow Method Step 2
N_COMPONENTS = 30   # hasil PCA Step 1 (90% variance)
RANDOM_SEED  = 42

# ── LOAD ─────────────────────────────────────────────────────────
print('Loading data ...')

df_pca50 = pd.read_csv(FILE_PCA50)
df_pca2  = pd.read_csv(FILE_PCA2)

print(f'  features_pca50 : {df_pca50.shape[0]:,} rows x {df_pca50.shape[1]} cols')
print(f'  features_pca2  : {df_pca2.shape[0]:,} rows x {df_pca2.shape[1]} cols')
print(f'  Kolom pertama  : {df_pca50.columns.tolist()[:5]}')

# Ambil ROW_ID sebagai identifier
# (Pipeline Phase 1 tidak menyimpan SK_ID_CURR, jadi kita pakai ROW_ID)
ids = df_pca50['ROW_ID'].values

# Ambil PC1 - PC30
pc_cols = [f'PC{i+1}' for i in range(N_COMPONENTS)]
pc_cols = [c for c in pc_cols if c in df_pca50.columns]

# float32 lebih hemat RAM daripada float64 (setengahnya)
X    = df_pca50[pc_cols].values.astype(np.float32)
X_2d = df_pca2[['PC1', 'PC2']].values

print(f'\n  Fitur dipakai : {pc_cols[0]} s/d {pc_cols[-1]} ({len(pc_cols)} komponen)')
print(f'  Memory X      : {X.nbytes / 1e6:.1f} MB')
print()
print('✅ Data siap!')

---
## Cell 3 — K-Means Clustering (K=5)

Pakai **MiniBatchKMeans** — lebih cepat dari KMeans biasa untuk data besar.  
Cara kerjanya sama, tapi tiap iterasi hanya proses subset (batch) data.

In [ ]:
print('=' * 55)
print(f'K-MEANS  (K={K_OPTIMAL})')
print('=' * 55)

start = time.time()

kmeans = MiniBatchKMeans(
    n_clusters   = K_OPTIMAL,
    batch_size   = 10_000,   # proses 10K rows per iterasi
    n_init       = 10,       # coba 10 inisialisasi, ambil terbaik
    max_iter     = 300,
    random_state = RANDOM_SEED,
    verbose      = 0,
)
kmeans_labels = kmeans.fit_predict(X)

elapsed = time.time() - start
print(f'  Selesai dalam : {elapsed:.1f} detik')
print(f'  Inertia       : {kmeans.inertia_:.2f}')
print()

# Distribusi applicant per cluster
unique, counts = np.unique(kmeans_labels, return_counts=True)
print('  Distribusi cluster:')
for cid, cnt in zip(unique, counts):
    pct = cnt / len(kmeans_labels) * 100
    bar = '█' * int(pct / 2)
    print(f'    Cluster {cid}: {cnt:>7,} rows ({pct:5.1f}%) {bar}')

# Silhouette score — pakai sample 10K biar cepat
print()
print('  Menghitung silhouette score (sample 10K) ...')
rng        = np.random.default_rng(RANDOM_SEED)
sample_idx = rng.choice(len(X), size=10_000, replace=False)
sil        = silhouette_score(X[sample_idx], kmeans_labels[sample_idx])
print(f'  Silhouette Score : {sil:.4f}')
print('  (0.1-0.3 wajar untuk data finansial kompleks)')
print()
print('✅ K-Means selesai!')

---
## Cell 4 — DBSCAN (Deteksi Outlier)

DBSCAN dijalankan pada **sample 30K** — full 355K terlalu berat di CPU.  
Tujuannya bukan clustering semua data, tapi **identifikasi profil outlier**.

**Parameter:**
- `eps` = radius neighborhood. Kalau outlier terlalu banyak (>20%) → naikkan eps
- `min_samples` = minimal titik dalam radius eps untuk jadi cluster

In [ ]:
print('=' * 55)
print('DBSCAN — Deteksi Outlier (sample 30K)')
print('=' * 55)

DBSCAN_SAMPLE = 30_000
EPS           = 3.0
MIN_SAMPLES   = 10

rng2       = np.random.default_rng(RANDOM_SEED)
dbscan_idx = rng2.choice(len(X), size=DBSCAN_SAMPLE, replace=False)
X_dbscan   = X[dbscan_idx]

print(f'  Sample size : {DBSCAN_SAMPLE:,} rows')
print(f'  Parameter   : eps={EPS}, min_samples={MIN_SAMPLES}')
print('  Running ... (1-3 menit)')

start         = time.time()
dbscan        = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, n_jobs=-1)
dbscan_labels = dbscan.fit_predict(X_dbscan)
elapsed       = time.time() - start

n_clusters_found = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_outliers       = int(np.sum(dbscan_labels == -1))
pct_outliers     = n_outliers / len(dbscan_labels) * 100

print(f'  Selesai dalam     : {elapsed:.1f} detik')
print(f'  Cluster ditemukan : {n_clusters_found}')
print(f'  Outlier (label -1): {n_outliers:,} ({pct_outliers:.1f}%)')
print()

unique_db, counts_db = np.unique(dbscan_labels, return_counts=True)
print('  Distribusi label DBSCAN:')
for label, cnt in zip(unique_db, counts_db):
    name = 'OUTLIER/NOISE' if label == -1 else f'Cluster {label}'
    pct  = cnt / len(dbscan_labels) * 100
    print(f'    {name:20s}: {cnt:>6,} ({pct:5.1f}%)')

print()
if pct_outliers < 5:
    print(f'  ✅ Outlier {pct_outliers:.1f}% — wajar, data cukup bersih')
elif pct_outliers < 20:
    print(f'  ⚠️  Outlier {pct_outliers:.1f}% — cukup banyak, masih acceptable')
else:
    print(f'  ❌ Outlier {pct_outliers:.1f}% — terlalu banyak!')
    print(f'     Coba ganti EPS = {EPS + 1.0} di atas lalu jalankan ulang cell ini')

print()
print('✅ DBSCAN selesai!')

---
## Cell 5 — Hierarchical Clustering + Dendrogram

Hierarchical standard butuh memori **O(n²)** → untuk 355K baris itu ~500 GB, mustahil di Kaggle.

Solusi: **Micro-cluster + scipy linkage** dalam 3 fase:

| Fase | GPU (cuML) | CPU fallback | Memori |
|------|-----------|--------------|--------|
| 1 | `cuML KMeans` → 500 micro-cluster centroids | `BIRCH partial_fit` → subcluster centers | O(n) |
| 2 | `scipy linkage (ward)` pada ≤500 centroids | sama | O(500²) kecil |
| 3 | `fcluster` cut + label assignment | `predict` per batch | O(batch) |

Inti algoritmanya tetap **hierarchical clustering** (`linkage` + `fcluster` + dendrogram).  
Fase 1 hanya membuat "wakil" data agar hierarchical bisa dijalankan pada seluruh 355K baris.  
GPU path jauh lebih cepat dan distribusi cluster lebih merata (KMeans lebih uniform dari BIRCH).

In [ ]:
# ── HIERARCHICAL CLUSTERING — GPU-accelerated (cuML + scipy) ─────
#
# Standard linkage: O(n²) memory → mustahil untuk 355K baris.
# Solusi: bikin "wakil" data terlebih dahulu (micro-clusters), lalu
# jalankan scipy hierarchical clustering pada ratusan centroids saja.
# Inti algoritmanya tetap hierarchical (linkage + fcluster + dendrogram).
#
# GPU path : cuML KMeans → 500 micro-cluster centroids (cepat di GPU)
# CPU path : BIRCH partial_fit → subcluster centers (fallback)

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

N_MICRO       = 500    # jumlah micro-cluster; lebih banyak = lebih detail
BATCH_SIZE_HC = 10_000

# ── Cek ketersediaan GPU (cuML) ───────────────────────────────────
try:
    from cuml.cluster import KMeans as cuKMeans
    import cupy as cp
    USE_GPU = True
except ImportError:
    from sklearn.cluster import Birch, AgglomerativeClustering
    USE_GPU = False

print('=' * 55)
print('HIERARCHICAL CLUSTERING')
print('=' * 55)
print(f'  Mode   : {"GPU — cuML KMeans + scipy linkage" if USE_GPU else "CPU — BIRCH + scipy linkage (fallback)"}')
print(f'  Data   : {len(X):,} rows (SELURUH DATA)')
if not USE_GPU:
    print('  ⚠️  cuML tidak tersedia — pastikan Accelerator = GPU di Settings Kaggle')
print()

n_rows = len(X)

if USE_GPU:
    # ── Fase 1: cuML KMeans → N_MICRO micro-cluster centroids (GPU) ──
    print(f'  Fase 1 — cuML KMeans ({N_MICRO} micro-clusters) ...')
    start  = time.time()
    X_gpu  = cp.asarray(X)
    kmicro = cuKMeans(n_clusters=N_MICRO, random_state=RANDOM_SEED, max_iter=300)
    kmicro.fit(X_gpu)
    micro_centers    = cp.asnumpy(kmicro.cluster_centers_)   # (N_MICRO, n_features)
    micro_labels_cpu = cp.asnumpy(kmicro.labels_).astype(int) # (n_rows,)
    elapsed_fase1    = time.time() - start
    print(f'  Fase 1 selesai : {elapsed_fase1:.1f} detik')

    # ── Fase 2: scipy linkage pada N_MICRO centroids (CPU, sangat cepat) ─
    print(f'\n  Fase 2 — scipy linkage pada {N_MICRO} centroids ...')
    linkage_methods = ['ward', 'complete', 'average']
    linkage_results = {}
    for method in linkage_methods:
        t0 = time.time()
        linkage_results[method] = linkage(micro_centers, method=method)
        print(f'    Linkage "{method:8s}" : {time.time()-t0:.1f} detik')

    # ── Fase 3: Potong dendrogram → assign semua rows ─────────────────
    # fcluster returns 1-indexed → konversi ke 0-indexed
    micro_to_cluster = fcluster(linkage_results['ward'], K_OPTIMAL, criterion='maxclust') - 1
    hier_labels      = micro_to_cluster[micro_labels_cpu].astype(np.int32)
    print(f'\n  Fase 3 — label assignment {n_rows:,} rows: selesai (instant)')

    X_dendro    = micro_centers
    dendro_note = f'{N_MICRO} micro-cluster centroids (GPU KMeans)  |  data penuh {n_rows:,} rows'
    elapsed_total = elapsed_fase1

else:
    # ── CPU FALLBACK: BIRCH + scipy linkage ───────────────────────────
    BIRCH_THRESHOLD = 0.3
    BIRCH_BF        = 50

    print(f'  Fase 1 — BIRCH partial_fit (threshold={BIRCH_THRESHOLD}) ...')
    start = time.time()
    birch = Birch(threshold=BIRCH_THRESHOLD, branching_factor=BIRCH_BF, n_clusters=None)
    for batch_start in range(0, n_rows, BATCH_SIZE_HC):
        birch.partial_fit(X[batch_start : batch_start + BATCH_SIZE_HC])
        batch_num = batch_start // BATCH_SIZE_HC + 1
        if batch_num % 5 == 0 or batch_start + BATCH_SIZE_HC >= n_rows:
            pct = min(batch_start + BATCH_SIZE_HC, n_rows) / n_rows * 100
            print(f'    Batch {batch_num:3d}: {pct:5.1f}%')
    elapsed_fase1 = time.time() - start
    subcenters    = birch.subcluster_centers_
    print(f'  Fase 1 selesai : {elapsed_fase1:.1f} detik  ({len(subcenters)} subclusters)')

    print(f'\n  Fase 2 — scipy linkage pada {len(subcenters)} subclusters ...')
    MAX_DENDRO = 500
    if len(subcenters) > MAX_DENDRO:
        rng3       = np.random.default_rng(RANDOM_SEED)
        dendro_idx = rng3.choice(len(subcenters), size=MAX_DENDRO, replace=False)
        X_dendro   = subcenters[dendro_idx]
    else:
        X_dendro   = subcenters

    linkage_methods = ['ward', 'complete', 'average']
    linkage_results = {}
    for method in linkage_methods:
        t0 = time.time()
        linkage_results[method] = linkage(X_dendro, method=method)
        print(f'    Linkage "{method:8s}" : {time.time()-t0:.1f} detik')

    # Agglomerative pada SEMUA subcenters untuk label assignment
    agglo = AgglomerativeClustering(n_clusters=K_OPTIMAL, linkage='ward')
    agglo.fit(subcenters)
    subcluster_to_cluster = agglo.labels_

    print(f'\n  Fase 3 — Assign {n_rows:,} rows ke cluster ...')
    start       = time.time()
    hier_labels = np.empty(n_rows, dtype=np.int32)
    for batch_start in range(0, n_rows, BATCH_SIZE_HC):
        batch   = X[batch_start : batch_start + BATCH_SIZE_HC]
        sub_ids = birch.predict(batch)
        hier_labels[batch_start : batch_start + BATCH_SIZE_HC] = subcluster_to_cluster[sub_ids]
    elapsed_fase3 = time.time() - start
    print(f'  Fase 3 selesai : {elapsed_fase3:.1f} detik')

    dendro_note   = f'{len(X_dendro)} subclusters BIRCH  |  data penuh {n_rows:,} rows'
    elapsed_total = elapsed_fase1 + elapsed_fase3

# ── Distribusi cluster ────────────────────────────────────────────
print()
unique_h, counts_h = np.unique(hier_labels, return_counts=True)
print('  Distribusi cluster Hierarchical:')
for cid, cnt in zip(unique_h, counts_h):
    pct = cnt / n_rows * 100
    bar = '█' * int(pct / 2)
    print(f'    Cluster {cid}: {cnt:>7,} rows ({pct:5.1f}%) {bar}')

# ── Dendrogram ────────────────────────────────────────────────────
print(f'\n  Dendrogram: {dendro_note}')
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors    = ['#2563EB', '#059669', '#DC2626']

for ax, method, color in zip(axes, linkage_methods, colors):
    dendrogram(
        linkage_results[method],
        ax                    = ax,
        truncate_mode         = 'lastp',
        p                     = 30,
        show_leaf_counts      = True,
        leaf_rotation         = 90,
        leaf_font_size        = 8,
        color_threshold       = 0,
        above_threshold_color = color,
    )
    ax.set_title(f'Linkage: {method}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Micro-cluster Index' if USE_GPU else 'Subcluster Index')
    ax.set_ylabel('Distance')

plt.suptitle(
    f'Hierarchical Dendrogram\n({dendro_note})',
    fontsize=12, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.savefig('/kaggle/working/dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print(f'  Total waktu : {elapsed_total:.1f} detik')
print(f'  ✅ Seluruh {n_rows:,} rows berhasil di-cluster secara hierarkis')
print()
print('✅ Hierarchical selesai!')

---
## Cell 6 — Visualisasi Scatter Plot 2D

In [ ]:
print('Membuat visualisasi scatter plot ...')

VIZ_SAMPLE = 20_000
rng4       = np.random.default_rng(RANDOM_SEED)
viz_idx    = rng4.choice(len(X_2d), size=VIZ_SAMPLE, replace=False)

COLORS = ['#2563EB', '#059669', '#DC2626', '#D97706', '#7C3AED']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── K-Means ───────────────────────────────────────────────────────
ax = axes[0]
for cid in range(K_OPTIMAL):
    mask = kmeans_labels[viz_idx] == cid
    ax.scatter(
        X_2d[viz_idx][mask, 0], X_2d[viz_idx][mask, 1],
        c=COLORS[cid], label=f'Cluster {cid}', alpha=0.4, s=5,
    )
ax.set_title(f'K-Means (K={K_OPTIMAL})', fontsize=13, fontweight='bold')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(markerscale=3)

# ── DBSCAN ────────────────────────────────────────────────────────
# DBSCAN hanya di sample 30K, viz dari subset itu
ax          = axes[1]
viz_db_size = min(VIZ_SAMPLE, len(dbscan_idx))
db_viz_idx  = dbscan_idx[:viz_db_size]
db_viz_lbl  = dbscan_labels[:viz_db_size]

for i, label in enumerate(sorted(set(db_viz_lbl))):
    mask  = db_viz_lbl == label
    name  = 'Outlier' if label == -1 else f'Cluster {label}'
    color = '#CCCCCC' if label == -1 else COLORS[min(i, len(COLORS)-1)]
    alpha = 0.2 if label == -1 else 0.5
    ax.scatter(
        X_2d[db_viz_idx][mask, 0], X_2d[db_viz_idx][mask, 1],
        c=color, label=name, alpha=alpha, s=5,
    )
ax.set_title(f'DBSCAN (eps={EPS}, min_samples={MIN_SAMPLES})', fontsize=13, fontweight='bold')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(markerscale=3)

plt.suptitle('Clustering Results — PCA 2D Projection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/clustering_viz.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('✅ Visualisasi selesai!')

---
## Cell 7 — Simpan Hasil

Di Kaggle, output disimpan ke `/kaggle/working/` — bisa didownload dari panel **Output** di kanan.

In [ ]:
print('=' * 55)
print('SIMPAN HASIL')
print('=' * 55)

# Gabungkan ROW_ID + hasil clustering ke satu DataFrame
df_results = pd.DataFrame({
    'ROW_ID'         : ids,
    'CLUSTER_KMEANS' : kmeans_labels,
    'CLUSTER_HIER'   : hier_labels,
})

# Tambahkan label DBSCAN untuk baris yang masuk sample
# Baris di luar sample → NaN
df_results['CLUSTER_DBSCAN'] = np.nan
df_results.loc[dbscan_idx, 'CLUSTER_DBSCAN'] = dbscan_labels.astype(float)

# Flag outlier: 1 = outlier DBSCAN, 0 = bukan, NaN = tidak di-sample
df_results['IS_OUTLIER'] = (df_results['CLUSTER_DBSCAN'] == -1).astype('Int8')

# Simpan ke /kaggle/working/ → bisa didownload dari panel Output
OUTPUT_PATH = '/kaggle/working/cluster_labels.csv'
df_results.to_csv(OUTPUT_PATH, index=False)

print(f'  File    : {OUTPUT_PATH}')
print(f'  Shape   : {df_results.shape[0]:,} rows x {df_results.shape[1]} cols')
print(f'  Kolom   : {df_results.columns.tolist()}')
print()
print('  Keterangan kolom:')
print('    CLUSTER_KMEANS  → K-Means (semua rows, K=5)')
print('    CLUSTER_HIER    → BIRCH + Agglomerative (semua rows, K=5)')
print('    CLUSTER_DBSCAN  → DBSCAN sample 30K (NaN = di luar sample)')
print('    IS_OUTLIER      → 1 jika DBSCAN label = -1')
print()
print('  Preview 5 baris pertama:')
print(df_results.head().to_string(index=False))
print()
print('=' * 55)
print('SEMUA OUTPUT TERSIMPAN DI /kaggle/working/')
print('=' * 55)
for f in os.listdir('/kaggle/working/'):
    size = os.path.getsize(f'/kaggle/working/{f}') / 1e6
    print(f'  {f}  ({size:.1f} MB)')
print()
print('Cara download:')
print('  → Lihat panel kanan → tab "Output"')
print('  → Klik file → Download')
print()
print('File yang dipindah ke laptop (datasets/final/):')
print('  cluster_labels.csv')
print('  clustering_viz.png')
print('  dendrogram.png')
print()
print('✅ Step 3 selesai! Lanjut Step 4 — Cluster Profiling di VS Code.')